In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 5 — Local Cover Letter Generator
## Generate Tailored Cover Letters from JD + Resume

**Stack:** Ollama · LangChain · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.4)

## Step 2 — Input Data

In [ ]:
resume_highlights = """
- 3 years Python backend (FastAPI, Django)
- Built microservices handling 10K RPM
- Led migration from monolith to microservices, reducing deploy time 60%
- Mentored 4 junior engineers through code reviews
- Experience with PostgreSQL, Redis, Docker, Kubernetes
"""

job_description = """
Senior Backend Engineer at CloudScale Inc.
- Build and maintain Python microservices (FastAPI)
- Design scalable APIs for 100K+ concurrent users
- Lead technical design reviews
- Mentor team of 5-8 engineers
- Experience with cloud deployments (AWS/GCP)
"""

company_info = """
CloudScale Inc. is a B2B SaaS company in the observability space.
They value engineering craftsmanship and open-source contributions.
Their tech blog emphasizes clean code and testing.
"""
print("Input data ready.")

## Step 3 — Multi-Paragraph Cover Letter Generation

In [ ]:
cover_letter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You write professional cover letters. Structure:
1. Opening — show enthusiasm + mention the role by name
2. Why you — 2 paragraphs connecting YOUR experience to THEIR needs
3. Why them — show you researched the company
4. Closing — call to action, thank them

Rules:
- Be specific, not generic. Reference actual projects and metrics.
- Match the company's tone (formal/startup-casual)
- Keep under 400 words
- Never fabricate experience not in the resume"""),
    ("human", """Write a cover letter for this role.

Resume highlights:
{resume}

Job Description:
{jd}

Company Info:
{company}

Generate the complete cover letter.""")
])

chain = cover_letter_prompt | llm | StrOutputParser()

letter = chain.invoke({
    "resume": resume_highlights,
    "jd": job_description,
    "company": company_info,
})
print(letter)

## Step 4 — Tone Variation Generator

In [ ]:
tone_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the following cover letter in a {tone} tone. Keep the core "
     "content but adjust the language, formality level, and energy."),
    ("human", "{letter}")
])
tone_chain = tone_prompt | llm | StrOutputParser()

for tone in ["confident and assertive", "humble and collaborative", "concise and technical"]:
    print(f"\n{'='*60}")
    print(f"TONE: {tone}")
    print('='*60)
    variant = tone_chain.invoke({"letter": letter, "tone": tone})
    print(variant[:500] + "...")

## Step 5 — Quality Check

In [ ]:
from pydantic import BaseModel, Field

class LetterReview(BaseModel):
    score: int = Field(description="Quality score 1-10")
    strengths: list[str]
    weaknesses: list[str]
    fabrication_risk: bool = Field(description="Does it claim experience not in resume?")

reviewer = llm.with_structured_output(LetterReview)
review = reviewer.invoke(
    f"""Review this cover letter against the resume.

Cover Letter: {letter}

Resume: {resume_highlights}

Score the letter and identify any fabricated claims."""
)
print(f"Score: {review.score}/10")
print(f"Fabrication Risk: {review.fabrication_risk}")
print(f"Strengths: {review.strengths}")
print(f"Weaknesses: {review.weaknesses}")

## What You Learned
- **Multi-section cover letter** generation from structured inputs
- **Tone variation** for different company cultures
- **Self-review pipeline** to catch fabrication and quality issues